# Project 38 — LangGraph Research Workflow with Memory
## Accumulate research findings over multiple iterations

**Stack:** LangGraph · Ollama · Jupyter

**Workflow:** Search → Evaluate → Accumulate → Synthesize

In [ ]:
# !pip install -q langgraph langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

class WorkflowState(TypedDict):
    input_data: str
    accumulated: str
    final_output: str

## Step 2 — Define Workflow Nodes

In [ ]:
def search(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 1 of a LangGraph Research Workflow with Memory pipeline. "
         "Your role: search. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[search]: " + result
    print(f"  Step 1/4: search")
    return {"accumulated": accumulated, "final_output": result}

def evaluate(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 2 of a LangGraph Research Workflow with Memory pipeline. "
         "Your role: evaluate. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[evaluate]: " + result
    print(f"  Step 2/4: evaluate")
    return {"accumulated": accumulated, "final_output": result}

def accumulate(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 3 of a LangGraph Research Workflow with Memory pipeline. "
         "Your role: accumulate. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[accumulate]: " + result
    print(f"  Step 3/4: accumulate")
    return {"accumulated": accumulated, "final_output": result}

def synthesize(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 4 of a LangGraph Research Workflow with Memory pipeline. "
         "Your role: synthesize. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[synthesize]: " + result
    print(f"  Step 4/4: synthesize")
    return {"accumulated": accumulated, "final_output": result}

## Step 3 — Build and Compile Graph

In [ ]:
graph = StateGraph(WorkflowState)
graph.add_node("search", search)
graph.add_node("evaluate", evaluate)
graph.add_node("accumulate", accumulate)
graph.add_node("synthesize", synthesize)

graph.set_entry_point("search")
graph.add_edge("search", "evaluate")
graph.add_edge("evaluate", "accumulate")
graph.add_edge("accumulate", "synthesize")
graph.add_edge("synthesize", END)

app = graph.compile()
print("LangGraph Research Workflow with Memory — workflow compiled!")

## Step 4 — Run the Workflow

In [ ]:
sample_input = "Analyze and process this request through the LangGraph Research Workflow with Memory pipeline."
result = app.invoke({
    "input_data": sample_input, "accumulated": "", "final_output": "",
})

print("=== WORKFLOW RESULT ===")
print(result["final_output"])
print("\n=== FULL TRACE ===")
print(result["accumulated"][:1000])

## What You Learned
- **Multi-node LangGraph workflow** for accumulate research findings over multiple iterations
- **Progressive context accumulation** across steps
- **Structured pipeline execution** with trace logging